https://github.com/heli2305/TriTueNhanTao

In [2]:
import tkinter as tk
from tkinter import ttk
from collections import deque
from dataclasses import dataclass
import heapq

# Du lieu ban dau
SIZE = 3
START_GRID = (
    (0, 0, 0),
    (1, 1, 0),
    (0, 0, 0),
)
START_POS = (0, 0)
GOAL_GRID = (
    (0, 0, 0),
    (0, 0, 0),
    (0, 0, 0),
)


@dataclass
class Node:
    name: str
    grid: tuple
    pos: tuple
    parent: object = None
    action: str = ""
    cost: int = 0


# Xu ly tim kiem
def is_goal(grid):
    for row in grid:
        for cell in row:
            if cell != 0:
                return False
    return True


def state_key(node):
    return (node.grid, node.pos)


def next_name(i):
    letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    if i < len(letters):
        return letters[i]
    return letters[i % len(letters)] + str(i // len(letters))


def valid_moves(pos):
    r, c = pos
    moves = []

    if r > 0:
        moves.append(("U", -1, 0))
    if r < SIZE - 1:
        moves.append(("D", 1, 0))
    if c > 0:
        moves.append(("L", 0, -1))
    if c < SIZE - 1:
        moves.append(("R", 0, 1))
    return moves


def make_child(node, move, name):
    action, dr, dc = move
    r, c = node.pos

    grid = [list(row) for row in node.grid]

    # Neu dang o o do thi hut sach truoc khi di chuyen
    if grid[r][c] == 1:
        grid[r][c] = 0

    nr, nc = r + dr, c + dc
    new_grid = tuple(tuple(row) for row in grid)

    return Node(
        name=name,
        grid=new_grid,
        pos=(nr, nc),
        parent=node,
        action=action,
        cost=node.cost + 1
    )


def path_to_root(node):
    path = []
    while node is not None:
        path.append(node)
        node = node.parent
    path.reverse()
    return path


def matrix_text(grid, pos=None):
    lines = []
    for r in range(SIZE):
        row = []
        for c in range(SIZE):
            if pos == (r, c):
                row.append("x")
            else:
                row.append(str(grid[r][c]))
        lines.append(" ".join(row))
    return "\n".join(lines)


def matrix_short(grid, pos=None):
    rows = []
    for r in range(SIZE):
        row = []
        for c in range(SIZE):
            if pos == (r, c):
                row.append("x")
            else:
                row.append(str(grid[r][c]))
        rows.append("".join(row))
    return "/".join(rows)


def is_cycle(node):
    current_key = state_key(node)
    parent = node.parent

    while parent is not None:
        if state_key(parent) == current_key:
            return True
        parent = parent.parent

    return False


def depth_limited_search(limit, version=1):
    start = Node("A", START_GRID, START_POS)
    frontier = [start]
    records = []
    expanded_names = []
    name_index = 1
    result = "failure"

    if is_goal(start.grid):
        records.append({
            "node_label": "A",
            "show_node": start,
            "frontier": [],
            "reached": [],
            "note": "Trang thai dau da la G"
        })
        return start, records

    while frontier:
        node = frontier.pop()
        expanded_names.append(node.name)

        if version == 1 and is_goal(node.grid):
            records.append({
                "node_label": node.name,
                "show_node": node,
                "frontier": list(frontier),
                "reached": list(expanded_names),
                "note": "Tim thay G"
            })
            return node, records

        if node.cost >= limit:
            result = "cutoff"
            records.append({
                "node_label": node.name,
                "show_node": node,
                "frontier": list(frontier),
                "reached": list(expanded_names),
                "note": f"Cham gioi han do sau {limit}"
            })
            continue

        if is_cycle(node):
            records.append({
                "node_label": node.name,
                "show_node": node,
                "frontier": list(frontier),
                "reached": list(expanded_names),
                "note": "Bo qua vi tao chu trinh"
            })
            continue

        found_child = None

        for move in reversed(valid_moves(node.pos)):
            child = make_child(node, move, next_name(name_index))
            name_index += 1

            if version == 2 and is_goal(child.grid):
                found_child = child
                break

            frontier.append(child)

        if found_child is not None:
            records.append({
                "node_label": f"{node.name} -> {found_child.name}",
                "show_node": found_child,
                "frontier": list(frontier),
                "reached": list(expanded_names),
                "note": f"Tim thay {found_child.name} khi sinh tu {node.name}"
            })
            return found_child, records

        records.append({
            "node_label": node.name,
            "show_node": node,
            "frontier": list(frontier),
            "reached": list(expanded_names),
            "note": "Da mo rong node"
        })

    return result, records


def iterative_deepening_search(version=1):
    depth = 0
    all_records = []

    while True:
        result, records = depth_limited_search(depth, version=version)

        if records:
            records[0]["reset_frontier_names"] = True
            records[0]["section"] = f"IDS cach {version} - depth = {depth}"
            all_records.extend(records)

        if isinstance(result, Node):
            return result, all_records, "success"

        if result == "failure":
            return None, all_records, "failure"

        depth += 1


def uniform_cost_search():
    start = Node("A", START_GRID, START_POS)
    frontier = [(start.cost, 0, start)]
    best_costs = {state_key(start): start.cost}
    records = []
    reached_names = []
    name_index = 1
    push_index = 1

    if is_goal(start.grid):
        records.append({
            "node_label": "A",
            "show_node": start,
            "frontier": [],
            "reached": [],
            "note": "Trang thai dau da la G"
        })
        return start, records, "success"

    while frontier:
        _, _, node = heapq.heappop(frontier)
        node_key = state_key(node)

        if node.cost > best_costs.get(node_key, float("inf")):
            continue

        reached_names.append(node.name)

        if is_goal(node.grid):
            records.append({
                "node_label": node.name,
                "show_node": node,
                "frontier": [item[2] for item in sorted(frontier)],
                "reached": list(reached_names),
                "note": "Tim thay G"
            })
            return node, records, "success"

        for move in valid_moves(node.pos):
            child = make_child(node, move, next_name(name_index))
            name_index += 1
            child_key = state_key(child)

            if child.cost < best_costs.get(child_key, float("inf")):
                best_costs[child_key] = child.cost
                heapq.heappush(frontier, (child.cost, push_index, child))
                push_index += 1

        records.append({
            "node_label": node.name,
            "show_node": node,
            "frontier": [item[2] for item in sorted(frontier)],
            "reached": list(reached_names),
            "note": "Da mo rong node"
        })

    return None, records, "failure"


def bfs_dfs_search(method, version=1):
    start = Node("A", START_GRID, START_POS)
    records = []
    reached_keys = set()
    reached_names = []
    name_index = 1

    if method == "BFS":
        frontier = deque([start])
    else:
        frontier = [start]

    if is_goal(start.grid):
        records.append({
            "node_label": "A",
            "show_node": start,
            "frontier": [],
            "reached": [],
            "note": "Trang thai dau da la G"
        })
        return start, records, "success"

    while frontier:
        node = frontier.popleft() if method == "BFS" else frontier.pop()
        reached_keys.add(state_key(node))
        reached_names.append(node.name)

        if version == 1 and is_goal(node.grid):
            records.append({
                "node_label": node.name,
                "show_node": node,
                "frontier": list(frontier),
                "reached": list(reached_names),
                "note": "Tim thay G"
            })
            return node, records, "success"

        found_child = None
        moves = valid_moves(node.pos)

        if method == "DFS":
            moves = list(reversed(moves))

        for move in moves:
            child = make_child(node, move, next_name(name_index))
            child_key = state_key(child)

            in_frontier = False
            for item in frontier:
                if state_key(item) == child_key:
                    in_frontier = True
                    break

            if child_key in reached_keys or in_frontier:
                continue

            name_index += 1

            if version == 2 and is_goal(child.grid):
                found_child = child
                break

            frontier.append(child)

        if found_child is not None:
            records.append({
                "node_label": f"{node.name} -> {found_child.name}",
                "show_node": found_child,
                "frontier": list(frontier),
                "reached": list(reached_names),
                "note": f"Tim thay {found_child.name} khi sinh tu {node.name}"
            })
            return found_child, records, "success"

        records.append({
            "node_label": node.name,
            "show_node": node,
            "frontier": list(frontier),
            "reached": list(reached_names),
            "note": "Da mo rong node"
        })

    return None, records, "failure"


def search(method, version=1):
    if method == "UCS":
        return uniform_cost_search()

    if method == "IDS":
        return iterative_deepening_search(version=version)

    return bfs_dfs_search(method, version=version)


# Giao dien

class VacuumApp(tk.Tk):
    def __init__(self):
        super().__init__()

        self.title("May hut bui - BFS, DFS, UCS, IDS")
        self.geometry("1540x900")
        self.minsize(770, 450)
        self.configure(bg="#f2f4f7")

        self.records = []
        self.goal_node = None
        self.search_status = "failure"
        self.current_title = ""
        self.after_id = None
        self.shown_frontier_names = set()

        self.make_ui()
        self.draw_start_screen()

    def make_ui(self):
        style = ttk.Style()
        style.theme_use("clam")
        style.configure("TButton", font=("Arial", 11, "bold"), padding=(10, 12))

        main = tk.Frame(self, bg="#f2f4f7", padx=16, pady=16)
        main.pack(fill="both", expand=True)

        main.grid_columnconfigure(1, weight=2)
        main.grid_columnconfigure(2, weight=0, minsize=600)
        main.grid_rowconfigure(0, weight=1)

        # Cot nut
        left = tk.Frame(main, bg="#f2f4f7")
        left.grid(row=0, column=0, sticky="ns", padx=(0, 16))

        ttk.Button(left, text="BFS cách 1", command=lambda: self.run_algo("BFS", 1)).pack(fill="x", pady=10)
        ttk.Button(left, text="BFS cách 2", command=lambda: self.run_algo("BFS", 2)).pack(fill="x", pady=10)
        ttk.Button(left, text="DFS cách 1", command=lambda: self.run_algo("DFS", 1)).pack(fill="x", pady=10)
        ttk.Button(left, text="DFS cách 2", command=lambda: self.run_algo("DFS", 2)).pack(fill="x", pady=10)
        ttk.Button(left, text="UCS", command=lambda: self.run_algo("UCS")).pack(fill="x", pady=10)
        ttk.Button(left, text="IDS cách 1", command=lambda: self.run_algo("IDS", 1)).pack(fill="x", pady=10)
        ttk.Button(left, text="IDS cách 2", command=lambda: self.run_algo("IDS", 2)).pack(fill="x", pady=10)

        # Cot giua
        middle = tk.Frame(main, bg="#f2f4f7")
        middle.grid(row=0, column=1, sticky="nsew")
        middle.grid_columnconfigure(0, weight=1)
        middle.grid_rowconfigure(0, weight=5)
        middle.grid_rowconfigure(1, weight=0, minsize=110)

        screen_box = tk.LabelFrame(
            middle,
            text="Màn hình minh họa hoạt động",
            font=("Arial", 12, "bold"),
            bg="white",
            padx=8,
            pady=8
        )
        screen_box.grid(row=0, column=0, sticky="nsew")
        screen_box.grid_columnconfigure(0, weight=1)
        screen_box.grid_rowconfigure(0, weight=1)

        self.canvas = tk.Canvas(screen_box, bg="white", highlightthickness=0)
        self.canvas.grid(row=0, column=0, sticky="nsew")

        result_box = tk.LabelFrame(
            middle,
            text="Kết quả chạy",
            font=("Arial", 12, "bold"),
            bg="white",
            height=180,
            padx=8,
            pady=8
        )
        result_box.grid(row=1, column=0, sticky="nsew", pady=(10, 0))
        result_box.grid_propagate(False)
        result_box.grid_columnconfigure(0, weight=1)
        result_box.grid_rowconfigure(0, weight=1)

        self.result_text = tk.Text(
            result_box,
            font=("Consolas", 10),
            wrap="word",
            state="disabled",
            bg="white",
            relief="flat"
        )
        self.result_text.grid(row=0, column=0, sticky="nsew")

        # Cot phai
        process_box = tk.LabelFrame(
            main,
            text="Quá trình các bước chạy",
            font=("Arial", 12, "bold"),
            bg="white",
            width=600,
            padx=8,
            pady=8
        )
        process_box.grid(row=0, column=2, sticky="nsew", padx=(16, 0))
        process_box.grid_propagate(False)
        process_box.grid_columnconfigure(0, weight=1)
        process_box.grid_rowconfigure(0, weight=1)

        self.process_text = tk.Text(
            process_box,
            width=62,
            font=("Consolas", 10),
            wrap="none",
            state="disabled",
            bg="white"
        )
        self.process_text.grid(row=0, column=0, sticky="nsew")

        yscroll = ttk.Scrollbar(process_box, orient="vertical", command=self.process_text.yview)
        yscroll.grid(row=0, column=1, sticky="ns")
        self.process_text.configure(yscrollcommand=yscroll.set)

    def draw_start_screen(self):
        self.canvas.delete("all")
        self.set_result(
            "S =\n" + matrix_text(START_GRID, START_POS) +
            "\n\nG =\n" + matrix_text(GOAL_GRID)
        )

    def set_result(self, text):
        self.result_text.config(state="normal")
        self.result_text.delete("1.0", "end")
        self.result_text.insert("end", text)
        self.result_text.config(state="disabled")

    def clear_process(self):
        self.shown_frontier_names = set()
        self.process_text.config(state="normal")
        self.process_text.delete("1.0", "end")
        self.process_text.insert(
            "end",
            f'{"Node":<8}| {"Frontier":<42}| Reached\n' +
            "-" * 8 + "+-" + "-" * 42 + "+-" + "-" * 18 + "\n"
        )
        self.process_text.config(state="disabled")

    def run_algo(self, method, version=1):
        if self.after_id is not None:
            self.after_cancel(self.after_id)
            self.after_id = None

        if method in {"BFS", "DFS", "IDS"}:
            self.current_title = f"{method} cách {version}"
            self.goal_node, self.records, self.search_status = search(method, version=version)
        else:
            self.current_title = "UCS"
            self.goal_node, self.records, self.search_status = search(method)

        self.clear_process()
        self.set_result("Đang chạy " + self.current_title + "...")
        self.show_step(0)

    def add_process_row(self, record):
        if record.get("reset_frontier_names"):
            self.shown_frontier_names = set()

        self.process_text.config(state="normal")
        if record.get("section"):
            self.process_text.insert("end", f"[{record['section']}]\n")
        self.process_text.config(state="disabled")

        frontier_lines = []

        for node in record["frontier"]:
            if node.name in self.shown_frontier_names:
                frontier_lines.append(node.name)
            else:
                parent_name = node.parent.name if node.parent else "-"
                action_name = node.action if node.action else "-"
                line = f"{node.name}: [{matrix_short(node.grid, node.pos)}], {parent_name}, {action_name}, {node.cost}"
                frontier_lines.append(line)
                self.shown_frontier_names.add(node.name)

        if not frontier_lines:
            frontier_lines = ["(rỗng)"]

        reached_text = "{" + ", ".join(record["reached"]) + "}" if record["reached"] else "{}"

        self.process_text.config(state="normal")
        for i, line in enumerate(frontier_lines):
            node_label = record["node_label"] if i == 0 else ""
            reached = reached_text if i == 0 else ""
            self.process_text.insert("end", f"{node_label:<8}| {line:<42}| {reached}\n")
        self.process_text.insert("end", "\n")
        self.process_text.see("end")
        self.process_text.config(state="disabled")

    def draw_grid(self, x, y, cell, grid, pos, label):
        self.canvas.create_text(x + cell * 1.5, y - 18, text=label, font=("Arial", 13, "bold"))

        for r in range(SIZE):
            for c in range(SIZE):
                left = x + c * cell
                top = y + r * cell
                value = grid[r][c]

                fill = "#fff7ed" if value == 1 else "#f8fafc"
                if pos == (r, c):
                    fill = "#bfdbfe" if value == 0 else "#fecaca"

                self.canvas.create_rectangle(
                    left, top, left + cell, top + cell,
                    fill=fill, outline="#111827", width=2
                )

                text = "x" if pos == (r, c) else str(value)
                self.canvas.create_text(
                    left + cell / 2,
                    top + cell / 2,
                    text=text,
                    font=("Arial", int(cell * 0.38), "bold"),
                    fill="#111827"
                )

    def draw_state(self, node, title, note):
        self.canvas.delete("all")

        small = 34
        big = 60

        start_x = 40
        current_x = 215
        goal_x = 540

        self.canvas.create_text(430, 32, text=title, font=("Arial", 18, "bold"), fill="#111827")

        self.draw_grid(start_x, 190, small, START_GRID, START_POS, "S")
        self.draw_grid(current_x, 120, big, node.grid, node.pos, node.name)
        self.draw_grid(goal_x, 190, small, GOAL_GRID, None, "G")

        self.canvas.create_text(
            current_x + big * 1.5,
            120 + big * 3 + 34,
            text="cost = " + str(node.cost),
            font=("Arial", 12, "bold"),
            fill="#374151"
        )

    def show_step(self, index):
        if index >= len(self.records):
            self.show_final_result()
            return

        record = self.records[index]
        self.draw_state(
            record["show_node"],
            f"{self.current_title} - bước {index + 1}",
            record["note"]
        )
        self.add_process_row(record)

        if index == len(self.records) - 1:
            self.after_id = self.after(900, self.show_final_result)
        else:
            self.after_id = self.after(900, lambda: self.show_step(index + 1))

    def show_final_result(self):
        self.after_id = None

        if self.goal_node is None:
            if self.search_status == "cutoff":
                self.set_result("Chưa tìm thấy lời giải trong giới hạn độ sâu đã chọn.")
            else:
                self.set_result("Không tìm thấy lời giải.")
            return

        path = path_to_root(self.goal_node)
        node_path = " -> ".join(node.name for node in path)
        action_path = " -> ".join(node.action for node in path[1:]) if len(path) > 1 else "(không có)"

        result = (
            "Tìm thấy lời giải bằng " + self.current_title + "\n"
            + "Đường đi node: " + node_path + "\n"
            + "Hành động: " + action_path + "\n"
            + "Tổng cost: " + str(self.goal_node.cost) + "\n\n"
            + "Trạng thái cuối:\n" + matrix_text(self.goal_node.grid, self.goal_node.pos)
        )
        self.set_result(result)


try:
    app.destroy()
except:
    pass

app = VacuumApp()
app.mainloop()